# Week 3b: Quantum Kernel Results — AG News (Dataset 2, N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**What this notebook does:**
1. Loads AG News embeddings from Week 2b
2. Computes 1000×1000 quantum kernel matrix (IQP, 8 qubits)
3. Compares quantum kernel SVM against classical baselines
4. Generates learning curves and comparison figures
5. Saves everything immediately to Drive

**Key difference from Week 3 (SST-2):**
- 4 classes (World / Sports / Business / Sci/Tech)
- Metric is **macro F1** throughout, not binary F1
- SVC handles multiclass automatically via one-vs-one

**Runtime:** ~83 min (same N=1000 kernel matrix)

In [ ]:
!pip install pennylane pennylane-lightning --quiet

import pennylane as qml
import numpy as np
import time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings('ignore')

print(f"PennyLane {qml.__version__}")
DEVICE = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage + checkpoint helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load AG News Week 2b data

In [ ]:
# Upload ag_news_embeddings.npz and ag_news_baseline_results.pkl
# OR load from Drive if already there

data = np.load(_find('ag_news_embeddings.npz'))
X_tr = data['X_tr_subset'].astype(np.float64)   # (1000, 8)
y_tr = data['y_tr_subset']
X_te = data['X_te_subset'].astype(np.float64)   # (200, 8)
y_te = data['y_te_subset']

AG_LABELS = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
N_CLASSES  = 4

with open(_find('ag_news_baseline_results.pkl'), 'rb') as f:
    baselines = pickle.load(f)

print(f"Train {X_tr.shape}  Test {X_te.shape}")
print(f"Value range: [{X_tr.min():.4f}, {X_tr.max():.4f}]  (must be in [0, pi])")
assert X_tr.shape == (1000, 8), "Expected (1000, 8) — re-run Week 2b"
assert X_te.shape == (200, 8),  "Expected (200, 8)  — re-run Week 2b"
print("N=1000 confirmed.")

print(f"\nClass balance (train):")
for i, name in AG_LABELS.items():
    print(f"  {name:<12}: {np.mean(y_tr==i):.1%}")

print(f"\nClassical baselines (from Week 2b):")
print(f"  {'Kernel':<14} {'Accuracy':>10} {'Macro-F1':>10}")
print("-"*38)
for k, v in baselines.items():
    print(f"  {k:<14} {v['accuracy']:>10.4f} {v['f1_macro']:>10.4f}")

---
## Step 2: IQP quantum kernel (same circuit as Week 3)

In [ ]:
N_QUBITS = 8

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(DEVICE)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])

# sanity check
assert np.isclose(quantum_kernel(X_tr[0], X_tr[0]), 1.0, atol=1e-5), "Kernel broken"
t0 = time.time()
for _ in range(3): quantum_kernel(X_tr[0], X_tr[1])
ms = (time.time()-t0)/3*1000
print(f"Sanity OK. Single eval {ms:.1f}ms")
print(f"N=1000 matrix estimate: ~{1000*1000*ms/1000/60:.0f} min")

---
## Step 3: Compute kernel matrix — saved immediately after

~83 min. Saved to Drive the moment it finishes.

In [ ]:
print(f"Computing {N_QUBITS}-qubit IQP kernel matrix (N=1000)...")
print("Saving immediately when done. Do not close tab.")
t0 = time.time()
K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min")

print("\nSaving kernel matrices immediately:")
checkpoint('ag_news_kernel_matrices.npz',
           lambda p: np.savez(p, K_train_quantum=K_train,
                              K_test_quantum=K_test, X_tr=X_tr, y_tr=y_tr))
print("\nSafe. Kernel matrices preserved.")

---
## Step 4: Quantum SVM + comparison

In [ ]:
qsvm   = SVC(kernel='precomputed', C=1.0)
qsvm.fit(K_train, y_tr)
y_pred = qsvm.predict(K_test)

acc_q    = float(accuracy_score(y_te, y_pred))
f1_q     = float(f1_score(y_te, y_pred, average='macro'))
f1_per_q = [float(f) for f in f1_score(y_te, y_pred, average=None)]

print(f"{'='*52}")
print(f"QUANTUM KERNEL SVM (N=1000, 8 qubits, IQP):")
print(f"  Accuracy    : {acc_q:.4f}")
print(f"  Macro F1    : {f1_q:.4f}")
print(f"  Per-class F1:")
for i, name in AG_LABELS.items():
    print(f"    {name:<12}: {f1_per_q[i]:.4f}")
print(f"{'='*52}\n")

print(f"vs classical baselines (N=1000):")
best_baseline_f1 = max(v['f1_macro'] for v in baselines.values())
for name, vals in baselines.items():
    diff_acc = acc_q - vals['accuracy']
    diff_f1  = f1_q  - vals['f1_macro']
    print(f"  vs {name:<12}: acc {diff_acc:+.4f}  macro-F1 {diff_f1:+.4f}")

---
## Step 5: Build results table (all N=1000 — fair comparison)

In [ ]:
results_all = {name: {'accuracy': v['accuracy'], 'f1_macro': v['f1_macro']}
               for name, v in baselines.items()}
results_all['Quantum (IQP)'] = {'accuracy': acc_q, 'f1_macro': f1_q}

print(f"{'Method':<16} {'Accuracy':>10} {'Macro-F1':>10}")
print("-"*38)
for name, v in results_all.items():
    marker = " <-- best F1" if v['f1_macro'] == max(r['f1_macro'] for r in results_all.values()) else ""
    print(f"{name:<16} {v['accuracy']:>10.4f} {v['f1_macro']:>10.4f}{marker}")

---
## Step 6: Comparison figure

In [ ]:
names  = list(results_all.keys())
accs   = [results_all[n]['accuracy'] for n in names]
f1s    = [results_all[n]['f1_macro'] for n in names]
colors = ['#5B8DB8', '#5B8DB8', '#5B8DB8', '#D95F02']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, vals, ylabel, title in [
    (axes[0], accs, 'Accuracy', f'Accuracy — AG News N=1000'),
    (axes[1], f1s,  'Macro F1', f'Macro F1 — AG News N=1000')]:
    ax.bar(names, vals, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
    ax.set_ylim(0.4, 0.9); ax.set_title(title); ax.set_ylabel(ylabel)
    for i, v in enumerate(vals):
        ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11, fontweight='500')
    ax.set_xticklabels(names, rotation=15, ha='right')

blue   = mpatches.Patch(color='#5B8DB8', label='Classical kernels')
orange = mpatches.Patch(color='#D95F02', label='Quantum IQP (8 qubits)')
fig.legend(handles=[blue, orange], loc='upper center', ncol=2,
           fontsize=11, bbox_to_anchor=(0.5, 1.02))
plt.suptitle('Quantum vs Classical SVM — 8-dim BERT Embeddings (AG News, 4-class)', y=1.06)
plt.tight_layout(); plt.show()
save_figure(fig, 'ag_news_fig1_comparison')

---
## Step 7: Learning curve

In [ ]:
train_sizes = [50, 100, 200, 400, 700, 1000]
lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}

print(f"{'N':>5}  {'Quantum':>9}  {'Linear':>9}  {'RBF':>9}  {'Time':>7}")
print("-"*46)
for n in train_sizes:
    t0 = time.time()
    # quantum — slice saved kernel matrix
    Kn      = K_train[:n, :n]
    Kn_test = K_test[:, :n]
    yp_q = SVC(kernel='precomputed', C=1.0).fit(Kn, y_tr[:n]).predict(Kn_test)
    lc['Quantum (IQP)'].append(float(f1_score(y_te, yp_q, average='macro')))
    # classical
    for cname, ck in [('Linear','linear'), ('RBF','rbf')]:
        clf = SVC(kernel=ck, C=1.0, gamma='scale' if ck=='rbf' else 1.0,
                  random_state=42).fit(X_tr[:n], y_tr[:n])
        lc[cname].append(float(f1_score(y_te, clf.predict(X_te), average='macro')))
    print(f"{n:>5}  {lc['Quantum (IQP)'][-1]:>9.4f}  "
          f"{lc['Linear'][-1]:>9.4f}  {lc['RBF'][-1]:>9.4f}  "
          f"{time.time()-t0:>6.1f}s")

fig2 = plt.figure(figsize=(8, 4))
cmap = {'Quantum (IQP)':'#D95F02','Linear':'#5B8DB8','RBF':'#2D9C4A'}
for m, vals in lc.items():
    ls = '-' if m=='Quantum (IQP)' else '--'
    lw = 2.5 if m=='Quantum (IQP)' else 1.8
    plt.plot(train_sizes, vals, label=m, color=cmap[m],
             linestyle=ls, linewidth=lw, marker='o', markersize=5)
plt.xlabel('Training set size (N)'); plt.ylabel('Macro F1')
plt.title('Learning curves — Macro F1 (AG News, 4-class)\n8-dim BERT embeddings')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()
save_figure(fig2, 'ag_news_fig2_learning_curves')

---
## Step 8: Cross-dataset comparison (SST-2 vs AG News)

In [ ]:
# Load SST-2 Week 3 results if available for side-by-side
sst2 = None
try:
    with open(_find('results_week3.json')) as f:
        sst2 = json.load(f)
    print("SST-2 results loaded for cross-dataset comparison.")
except:
    print("SST-2 results not found — skipping cross-dataset plot.")
    print("Copy results_week3.json to Drive and re-run to enable.")

if sst2:
    methods = ['Linear', 'RBF', 'Polynomial', 'Quantum (IQP)']
    sst2_accs = [sst2['all_results'][m]['accuracy'] for m in methods]
    ag_accs   = [results_all.get(m, results_all.get(m.split(' ')[0], {'accuracy':0}))['accuracy'] for m in methods]
    # rebuild cleanly
    sst2_vals = [sst2['all_results'][k]['accuracy'] for k in ['Linear','RBF','Polynomial','Quantum (IQP)']]
    ag_vals   = [results_all[k]['accuracy'] for k in results_all]

    fig3, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(names)); w = 0.35
    ax.bar(x-w/2, sst2_vals, w, label='SST-2 (sentiment, binary)',
           color=['#5B8DB8','#5B8DB8','#5B8DB8','#D95F02'], edgecolor='white', alpha=0.9)
    ax.bar(x+w/2, [results_all[n]['accuracy'] for n in names], w,
           label='AG News (topic, 4-class)',
           color=['#2D9C4A','#2D9C4A','#2D9C4A','#FF6B35'], edgecolor='white', alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylim(0.4, 0.9); ax.set_ylabel('Accuracy')
    ax.set_title('Cross-dataset comparison: SST-2 vs AG News (N=1000)')
    ax.legend()
    plt.tight_layout(); plt.show()
    save_figure(fig3, 'cross_dataset_fig_comparison')

---
## Step 9: Save results JSON

In [ ]:
results_week3b = {
    'dataset': 'ag_news',
    'n_classes': N_CLASSES,
    'class_names': AG_LABELS,
    'n_qubits': N_QUBITS,
    'n_train': int(len(X_tr)),
    'n_test': int(len(X_te)),
    'feature_map': 'IQP (Hadamard + RZ + ZZ)',
    'compute_time_min': round(elapsed/60, 1),
    'quantum_accuracy': acc_q,
    'quantum_f1_macro': f1_q,
    'quantum_f1_per_class': {AG_LABELS[i]: f1_per_q[i] for i in range(N_CLASSES)},
    'all_results': results_all,
    'learning_curve_sizes': train_sizes,
    'learning_curve_f1_macro': {k: v for k, v in lc.items()},
}
checkpoint('ag_news_results_week3b.json',
           lambda p: json.dump(results_week3b, open(p,'w'), indent=2))

print("\n=== WEEK 3b DONE ===")
print(f"Dataset       : AG News (4-class topic)")
print(f"Quantum IQP   : acc={acc_q:.4f}  macro-F1={f1_q:.4f}")
print(f"Best classical: acc={max(v['accuracy'] for v in baselines.values()):.4f}  "
      f"macro-F1={max(v['f1_macro'] for v in baselines.values()):.4f}")
print(f"\nFiles saved:")
ag_files = [f for f in sorted(os.listdir(SAVE_DIR)) if 'ag_news' in f or 'cross' in f]
for f in ag_files:
    print(f"  {f}")

---
## Week 3b complete

### Saved + downloaded:
- `ag_news_kernel_matrices.npz` — saved immediately after the 83-min compute
- `ag_news_fig1_comparison.png/.pdf` — accuracy and macro F1 bars
- `ag_news_fig2_learning_curves.png/.pdf` — macro F1 vs training size
- `cross_dataset_fig_comparison.png/.pdf` — SST-2 vs AG News side-by-side
- `ag_news_results_week3b.json` — all numbers

### Cross-dataset picture (for the paper):
With both SST-2 and AG News results you can now make the core claim:
quantum kernels behave consistently across NLP tasks and domains.
The learning curve pattern (quantum needs more data to converge) should
replicate on AG News — if it does, that is a genuine cross-domain finding.

### Next:
- Week 2c: CLIP + CIFAR-10 (vision, Dataset 3)
- Week 4b: Ablations on AG News (same as Week 4 but loading ag_news_embeddings.npz)
- Or: Week 5 analysis once you have both NLP datasets done